# Querying the Semantic Layer

Instead of writing SQL, SLayer queries describe **what** data you want: measures, dimensions, filters, and time granularity. SLayer generates and executes the SQL.

This notebook demonstrates common query patterns against the Jaffle Shop dataset, which has been auto-ingested with rollup joins across 7 tables.

See also: [Queries docs](../../concepts/queries.md) | [Formulas docs](../../concepts/formulas.md)

**Prerequisites:** `pip install motley-slayer[examples]`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

/home/james/miniconda3/envs/motley3.11/lib/python3.11/site-packages/sqlglot/tokens.py:14: UserWarning: sqlglot[rs] is deprecated and no longer compatible with sqlglot. Please use sqlglotc instead for faster parsing: pip install sqlglot[c]
  warnings.warn(


## Revenue by Store

The `orders` model has a join to `stores`, so we can group by `stores.name` directly — SLayer resolves the JOIN automatically:

In [2]:
result = engine.execute(query={
    "source_model": "orders",
    "fields": [{"formula": "count"}, {"formula": "order_total_sum"}],
    "dimensions": [{"name": "stores.name"}],
    "order": [{"column": {"name": "order_total_sum"}, "direction": "desc"}],
})

for row in result.data:
    store = row["orders.stores.name"]
    count = row["orders.count"]
    revenue = row["orders.order_total_sum"]
    print(f"  {store}: {count:,} orders, ${revenue:,.2f} revenue")

  Brooklyn: 80,349 orders, $1,014,796.82 revenue
  Philadelphia: 58,293 orders, $753,880.60 revenue
  Chicago: 34,997 orders, $452,218.31 revenue
  San Francisco: 28,453 orders, $373,174.21 revenue
  New Orleans: 4,293 orders, $55,509.93 revenue


## Monthly Order Trends with Cumulative Sum

Time dimensions truncate a date/timestamp column to a given granularity. The `cumsum` formula function computes a running total over time:

In [3]:
result = engine.execute(query={
    "source_model": "orders",
    "time_dimensions": [{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    "fields": [
        {"formula": "count"},
        {"formula": "cumsum(count)", "name": "cumulative"},
    ],
    "order": [{"column": {"name": "ordered_at"}, "direction": "asc"}],
})

print(f"{'Month':<12} {'Orders':>8} {'Cumulative':>12}")
print("-" * 34)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    count = row["orders.count"]
    cumulative = row["orders.cumulative"]
    print(f"{month:<12} {count:>8,} {cumulative:>12,}")

Month          Orders   Cumulative
----------------------------------
2018-09           379          379
2018-10           654        1,033
2018-11           874        1,907
2018-12         1,117        3,024
2019-01         1,429        4,453
2019-02         1,387        5,840
2019-03         2,404        8,244
2019-04         2,905       11,149
2019-05         3,177       14,326
2019-06         2,666       16,992
2019-07         1,649       18,641
2019-08         1,812       20,453
2019-09         2,725       23,178
2019-10         5,180       28,358
2019-11         5,114       33,472
2019-12         5,683       39,155
2020-01         5,886       45,041
2020-02         5,428       50,469
2020-03         5,885       56,354
2020-04         5,749       62,103
2020-05         7,595       69,698
2020-06         6,155       75,853
2020-07         3,505       79,358
2020-08         3,471       82,829
2020-09         5,971       88,800
2020-10        10,793       99,593
2020-11        10,88

## Top Products by Quantity Sold

This query uses the `order_items` model, which has a join to `products`. The joined dimensions `products.name` and `products.type` are available automatically:

In [4]:
result = engine.execute(query={
    "source_model": "order_items",
    "fields": [{"formula": "count"}, {"formula": "quantity_sum"}],
    "dimensions": [{"name": "products.name"}, {"name": "products.type"}],
    "order": [{"column": {"name": "quantity_sum"}, "direction": "desc"}],
})

print(f"{'Product':<35} {'Type':<10} {'Line Items':>12} {'Qty Sold':>10}")
print("-" * 70)
for row in result.data:
    name = row["order_items.products.name"]
    ptype = row["order_items.products.type"]
    count = row["order_items.count"]
    qty = row["order_items.quantity_sum"]
    print(f"{name:<35} {ptype:<10} {count:>12,} {qty:>10,}")

Product                             Type         Line Items   Qty Sold
----------------------------------------------------------------------
tangaroo                            BEVERAGE         46,450     48,232
for richer or pourover              BEVERAGE         46,163     47,943
adele-ade                           BEVERAGE         46,202     47,896
vanilla ice                         BEVERAGE         46,142     47,870
chai and mighty                     BEVERAGE         46,004     47,675
the krautback                       JAFFLE           18,781     19,658
nutellaphone who dis?               JAFFLE           18,763     19,613
flame impala                        JAFFLE           18,740     19,545
mel-bun                             JAFFLE           18,632     19,418
doctor stew                         JAFFLE           18,538     19,359


## Month-over-Month Revenue Change

The `change()` and `change_pct()` formula functions compute the difference from the previous time period. They use self-join CTEs internally, so they handle gaps correctly and don't produce edge NULLs:

In [5]:
result = engine.execute(query={
    "source_model": "orders",
    "time_dimensions": [{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    "fields": [
        {"formula": "order_total_sum"},
        {"formula": "change(order_total_sum)", "name": "mom_change"},
        {"formula": "change_pct(order_total_sum)", "name": "mom_pct"},
    ],
    "order": [{"column": {"name": "ordered_at"}, "direction": "asc"}],
    "limit": 12,
})

print(f"{'Month':<12} {'Revenue':>12} {'MoM Change':>12} {'MoM %':>8}")
print("-" * 48)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    pct = row["orders.mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>11,.2f} {chg_str:>12} {pct_str:>8}")

Month             Revenue   MoM Change    MoM %
------------------------------------------------
2018-09      $   4,738.80            -        -
2018-10      $   7,882.42      $+3,144   +66.3%
2018-11      $  11,394.11      $+3,512   +44.6%
2018-12      $  14,904.61      $+3,511   +30.8%
2019-01      $  17,790.62      $+2,886   +19.4%
2019-02      $  17,464.28        $-326    -1.8%
2019-03      $  31,275.57     $+13,811   +79.1%
2019-04      $  36,448.96      $+5,173   +16.5%
2019-05      $  40,419.76      $+3,971   +10.9%
2019-06      $  34,389.80      $-6,030   -14.9%
2019-07      $  21,901.22     $-12,489   -36.3%
2019-08      $  24,826.53      $+2,925   +13.4%


## Filtering

Filters are condition strings that support SQL comparison operators (`=`, `<>`, `>`, `>=`, `<`, `<=`), `IN`, `IS NULL`, `LIKE`, and boolean logic (`AND`, `OR`). Filters on joined dimensions work the same way:

In [6]:
result = engine.execute(query={
    "source_model": "orders",
    "fields": [{"formula": "count"}, {"formula": "order_total_sum"}],
    "dimensions": [{"name": "stores.name"}],
    "filters": ["stores.name in ('Brooklyn', 'Philadelphia')"],
    "order": [{"column": {"name": "order_total_sum"}, "direction": "desc"}],
})

for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders.count']:,} orders, ${row['orders.order_total_sum']:,.2f}")

  Brooklyn: 80,349 orders, $1,014,796.82
  Philadelphia: 58,293 orders, $753,880.60


## Arithmetic Fields

Field formulas can combine measures with arithmetic. Use `name` to give the computed column a short alias:

In [7]:
result = engine.execute(query={
    "source_model": "orders",
    "fields": [
        {"formula": "count"},
        {"formula": "order_total_sum"},
        {"formula": "order_total_sum / count", "name": "avg_order_value"},
    ],
    "dimensions": [{"name": "stores.name"}],
    "order": [{"column": {"name": "avg_order_value"}, "direction": "desc"}],
})

print(f"{'Store':<20} {'Orders':>8} {'Revenue':>14} {'Avg Order':>12}")
print("-" * 58)
for row in result.data:
    print(f"{row['orders.stores.name']:<20} {row['orders.count']:>8,} ${row['orders.order_total_sum']:>13,.2f} ${row['orders.avg_order_value']:>11,.2f}")

Store                  Orders        Revenue    Avg Order
----------------------------------------------------------
San Francisco          28,453 $   373,174.21 $      13.12
Philadelphia           58,293 $   753,880.60 $      12.93
New Orleans             4,293 $    55,509.93 $      12.93
Chicago                34,997 $   452,218.31 $      12.92
Brooklyn               80,349 $ 1,014,796.82 $      12.63


## Summary

SLayer queries are declarative — you describe what data you want, not how to get it:

- **Dimensions** group the data (including joined dimensions like `stores.name`)
- **Fields** define computed columns: measures, arithmetic, and transform functions
- **Time dimensions** add time-based grouping with truncation granularity
- **Filters** restrict rows using familiar comparison syntax

JOINs, GROUP BY, and window functions are all generated automatically.

**Next:** See the [Joins](../joins/joins.ipynb) notebook for a deep dive into how joins work in SLayer.